# Colab 01 — Dense embeddings και cross-encoder

Τρέχει δύο πειράματα της πτυχιακής στη GPU του Colab, με τον ίδιο κώδικα και τα ίδια splits όπως τοπικά:
1. **Dense embeddings** (χωρίς εκπαίδευση): `dense_minilm`, `dense_bge`
2. **Fine-tuned cross-encoder** για E/S/C/I: `cross_encoder_minilm` (~1 ώρα σε T4)

**Πριν ξεκινήσεις:** Runtime → Change runtime type → **T4 GPU**. Μετά **Runtime → Run all**.
Στο τέλος κατεβαίνει το `results_colab01.zip`. Δώσ' το στο Claude για να μπει στο repo.

Τίποτα δεν εγκαθίσταται στον υπολογιστή σου· όλα γίνονται στο μηχάνημα της Google.

## 1. Έλεγχος GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Κώδικας από το GitHub και πακέτα

In [ ]:
%cd /content
!rm -rf repo && git clone --depth 1 https://github.com/CharalamposZisis/Amazon-Recommendation-System.git repo
%cd /content/repo
!pip install -q sentence-transformers pyyaml

## 3. ESCI (~2GB, στο Colab) και δημιουργία splits
Ίδιο seed (42) με τα τοπικά splits. Το κελί ελέγχει ότι τα μεγέθη βγαίνουν ακριβώς ίδια.

In [ ]:
RAW = "data/esci_data/esci-data/shopping_queries_dataset"
!mkdir -p {RAW}
!wget -q -nc -P {RAW} https://media.githubusercontent.com/media/amazon-science/esci-data/main/shopping_queries_dataset/shopping_queries_dataset_examples.parquet
!wget -q -nc -P {RAW} https://media.githubusercontent.com/media/amazon-science/esci-data/main/shopping_queries_dataset/shopping_queries_dataset_products.parquet
!wget -q -nc -P {RAW} https://raw.githubusercontent.com/amazon-science/esci-data/main/shopping_queries_dataset/shopping_queries_dataset_sources.csv
!ls -la {RAW}
!python -m src.data --config configs/data.yaml

In [ ]:
import json
info = json.load(open("data/processed/esci_small_us/split_info.json"))["splits"]
expected = {"train": (377719, 18799), "val": (41934, 2089), "test": (None, 8956)}
for s, (rows, queries) in expected.items():
    assert info[s]["queries"] == queries, (s, info[s])
    assert rows is None or info[s]["rows"] == rows, (s, info[s])
print("Τα splits είναι ίδια με τα τοπικά ✔")

## 4. Smoke test (λίγα λεπτά)
Τρέχει όλα σε 100 queries για να φανεί αμέσως αν κάτι δεν δουλεύει, πριν τα μεγάλα τρεξίματα.

In [ ]:
!python -m src.dense --config configs/dense_minilm.yaml --max-queries 100
!python -m src.cross_encoder --config configs/cross_encoder_minilm.yaml --max-queries 100
!rm -rf results/*_smoke

## 5. Dense embeddings (~10–20 λεπτά)

In [ ]:
!python -m src.dense --config configs/dense_minilm.yaml
!python -m src.dense --config configs/dense_bge.yaml

## 6. Cross-encoder (~1 ώρα σε T4)
Αν το Colab αποσυνδεθεί, ξανατρέξε από το βήμα 2· τα αποτελέσματα του βήματος 5 θα χρειαστεί να ξαναβγούν.

In [ ]:
!python -m src.cross_encoder --config configs/cross_encoder_minilm.yaml

## 7. Πίνακας και κατέβασμα αποτελεσμάτων

In [ ]:
!python -m src.report --split test
!zip -r -q results_colab01.zip results/dense_minilm results/dense_bge results/cross_encoder_minilm -x "*.pt"
from google.colab import files
files.download("results_colab01.zip")